## Find the file path

In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/"
    )
)

## Read CSV Filse

In [0]:
df_customer = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/DimCustomer.csv")
    
display(df_customer)

In [0]:
df_product = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/DimProduct.csv"
)

df_date = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/DimDate.csv"
)

df_geography = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/DimGeography.csv"
)

df_sales = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/FactInternetSales.csv"
)

print("All SQL Bronze DataFrames loaded successfully")


## Read API Bronze data.

In [0]:
df_api = spark.read.option("multiline", "true").json("abfss://bronze@adventureworkstorageadls.dfs.core.windows.net/API/frankfurter_usd_inr.json")

display(df_api)

## Clean & transform SQL data

In [0]:
from pyspark.sql.functions import col, trim

dm_customer_clean = df_customer.select(
                            *[trim(col(c)).alias(c) for c in df_customer.columns])

                           
display(dm_customer_clean)

## Make dynamic approach

In [0]:
from pyspark.sql.functions import col, trim

def clean_dataframe(df):
    return df.select(
        *[
            trim(col(c)).alias(c)
            for c in df.columns
        ]
    )
sql_dataframes = {
    "DimCustomer": df_customer,
    "DimProduct": df_product,
    "DimDate": df_date,
    "DimGeography": df_geography,
    "FactInternetSales": df_sales
}

cleaned_dataframes = {name: clean_dataframe(df) for name, df in sql_dataframes.items()}

df_customer_clean = cleaned_dataframes["DimCustomer"]
df_product_clean = cleaned_dataframes["DimProduct"]
df_date_clean = cleaned_dataframes["DimDate"]
df_geography_clean = cleaned_dataframes["DimGeography"]
df_sales_clean = cleaned_dataframes["FactInternetSales"]

Clean & transform API data

In [0]:
from pyspark.sql.functions import col

df_api_clean = df_api.select(
    col("amount").cast("double").alias("amount"),
    col("base").alias("base_currency"),
    col("date").alias("exchange_date"),
    col("rates.INR").cast("double").alias("INR_rate")
)

display(df_api_clean)

## Handling Nulls

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

for table_name, df in sql_dataframes.items():
    print(f"\n--- {table_name} ---")
    
    null_counts = df.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])
    
    display(null_counts)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, lit

null_results = []

for table_name, df in sql_dataframes.items():
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        null_results.append((table_name, c, null_count))

null_df = spark.createDataFrame(
    null_results,
    ["TableName", "ColumnName", "NullCount"]
)

display(null_df.orderBy("TableName", "ColumnName"))

## Silver data-quality transformation

1.Trim string columns.

2.Preserve NULL values.

3.Keep numeric/date NULLs as NULL.

4.Remove completely empty rows if any.

In [0]:
def clean_dataframe(df):
    # Trim whitespace from string columns while preserving NULLs
    for field in df.schema.fields:
        if field.dataType.simpleString() == "string":
            df = df.withColumn(field.name, trim(col(field.name)))

    # Remove completely empty rows
    df = df.dropna(how="all")

    return df


cleaned_dataframes = {
    name: clean_dataframe(df)
    for name, df in sql_dataframes.items()
}

print("Silver transformations completed for all SQL tables.")

In [0]:
for table_name, df in cleaned_dataframes.items():
    print(f"\n--- {table_name} ---")

    null_results = [
        (
            table_name,
            c,
            df.filter(col(c).isNull()).count()
        )
        for c in df.columns
    ]

    display(
        spark.createDataFrame(
            null_results,
            ["TableName", "ColumnName", "NullCount"]
        )
    )

In [0]:
df_product_clean.printSchema()

In [0]:
from pyspark.sql.functions import col

df_product_silver = df_product_clean.select(
    col("ProductKey").cast("int").alias("ProductKey"),
    col("ProductAlternateKey"),
    col("ProductSubcategoryKey").cast("int").alias("ProductSubcategoryKey"),
    col("WeightUnitMeasureCode"),
    col("SizeUnitMeasureCode"),
    col("EnglishProductName"),
    col("SpanishProductName"),
    col("FrenchProductName"),
    col("StandardCost").cast("decimal(18,2)").alias("StandardCost"),
    col("FinishedGoodsFlag").cast("boolean").alias("FinishedGoodsFlag"),
    col("Color"),
    col("SafetyStockLevel").cast("int").alias("SafetyStockLevel"),
    col("ReorderPoint").cast("int").alias("ReorderPoint"),
    col("ListPrice").cast("decimal(18,2)").alias("ListPrice"),
    col("Size"),
    col("SizeRange"),
    col("Weight").cast("decimal(18,2)").alias("Weight"),
    col("DaysToManufacture").cast("int").alias("DaysToManufacture"),
    col("ProductLine"),
    col("DealerPrice").cast("decimal(18,2)").alias("DealerPrice"),
    col("Class"),
    col("Style"),
    col("ModelName"),
    col("LargePhoto"),
    col("EnglishDescription"),
    col("FrenchDescription"),
    col("ChineseDescription"),
    col("ArabicDescription"),
    col("HebrewDescription"),
    col("ThaiDescription"),
    col("GermanDescription"),
    col("JapaneseDescription"),
    col("TurkishDescription"),
    col("StartDate").cast("date").alias("StartDate"),
    col("EndDate").cast("date").alias("EndDate"),
    col("Status")
)

df_product_silver.printSchema()

In [0]:
display(df_product_silver)

In [0]:
df_product_silver.select(
    "ProductKey",
    "StandardCost",
    "ListPrice",
    "DealerPrice",
    "Weight",
    "StartDate",
    "EndDate"
).describe().display()

## metadata-driven type-mapping layer.

In [0]:
type_mapping = {
    "DimCustomer": {
        "CustomerKey": "int",
        "GeographyKey": "int",
        "NameStyle": "boolean",
        "HouseOwnerFlag": "boolean",
        "NumberCarsOwned": "int",
        "NumberChildrenAtHome": "int",
        "TotalChildren": "int",
        "YearlyIncome": "decimal(18,2)",
        "DateFirstPurchase": "date",
        "BirthDate": "date"
    },

    "DimDate": {
        "DateKey": "int",
        "CalendarQuarter": "int",
        "CalendarSemester": "int",
        "CalendarYear": "int",
        "DayNumberOfMonth": "int",
        "DayNumberOfWeek": "int",
        "DayNumberOfYear": "int",
        "FiscalQuarter": "int",
        "FiscalSemester": "int",
        "FiscalYear": "int",
        "MonthNumberOfYear": "int",
        "WeekNumberOfYear": "int",
        "FullDateAlternateKey": "date"
    },

    "DimGeography": {
        "GeographyKey": "int",
        "SalesTerritoryKey": "int"
    },

    "FactInternetSales": {
        "CurrencyKey": "int",
        "CustomerKey": "int",
        "DueDateKey": "int",
        "OrderDateKey": "int",
        "ProductKey": "int",
        "PromotionKey": "int",
        "RevisionNumber": "int",
        "SalesOrderLineNumber": "int",
        "SalesTerritoryKey": "int",
        "ShipDateKey": "int",
        "OrderQuantity": "int",
        "DiscountAmount": "decimal(18,2)",
        "ExtendedAmount": "decimal(18,2)",
        "Freight": "decimal(18,2)",
        "ProductStandardCost": "decimal(18,2)",
        "SalesAmount": "decimal(18,2)",
        "TaxAmt": "decimal(18,2)",
        "UnitPrice": "decimal(18,2)",
        "UnitPriceDiscountPct": "decimal(18,4)"
    }
}


In [0]:
def apply_type_mapping(df, table_name):
    mappings = type_mapping.get(table_name, {})

    for column_name, data_type in mappings.items():
        if column_name in df.columns:
            df = df.withColumn(
                column_name,
                col(column_name).cast(data_type)
            )

    return df

In [0]:
cleaned_dataframes = {
    table_name: apply_type_mapping(df, table_name)
    for table_name, df in sql_dataframes.items()
}

for table_name, df in cleaned_dataframes.items():
    print(f"\n--- {table_name} ---")
    df.printSchema()

In [0]:
type_mapping["DimProduct"] = {
    "ProductKey": "int",
    "ProductSubcategoryKey": "int",
    "StandardCost": "decimal(18,2)",
    "FinishedGoodsFlag": "boolean",
    "SafetyStockLevel": "int",
    "ReorderPoint": "int",
    "ListPrice": "decimal(18,2)",
    "Weight": "decimal(18,2)",
    "DaysToManufacture": "int",
    "DealerPrice": "decimal(18,2)",
    "StartDate": "date",
    "EndDate": "date"
}

print("DimProduct type mapping added")

In [0]:
type_mapping["DimProduct"].update({
    "StandardCost": "decimal(18,2)",
    "ListPrice": "decimal(18,2)",
    "DealerPrice": "decimal(18,2)",
    "Weight": "decimal(18,2)",
    "StartDate": "date",
    "EndDate": "date"
})

type_mapping["FactInternetSales"].update({
    "TotalProductCost": "decimal(18,2)",
    "OrderDate": "date",
    "DueDate": "date",
    "ShipDate": "date"
})

In [0]:
silver_dataframes = {
    table_name: apply_type_mapping(df, table_name)
    for table_name, df in sql_dataframes.items()
}

for table_name, df in silver_dataframes.items():
    print(f"\n--- {table_name} ---")
    df.printSchema()

# write the Silver data as Parquet

In [0]:
silver_base = "abfss://silver@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025"

for table_name, df in silver_dataframes.items():
    (
        df.write
        .mode("overwrite")
        .parquet(f"{silver_base}/{table_name}")
    )

print("All SQL Silver data written successfully as Parquet")

## write the API Silver data as Parquet

In [0]:
api_silver_path = "abfss://silver@adventureworkstorageadls.dfs.core.windows.net/API/Frankfurter"

df_api_clean.write \
    .mode("overwrite") \
    .parquet(api_silver_path)

print("API Silver data written successfully as Parquet")

## Validate SQL Silver

In [0]:
for table_name in silver_dataframes.keys():
    path = f"abfss://silver@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025/{table_name}"
    
    df_check = spark.read.parquet(path)
    
    print(f"{table_name}: {df_check.count()} rows")

## validate API Silver


In [0]:
api_check_path = "abfss://silver@adventureworkstorageadls.dfs.core.windows.net/API/Frankfurter"

df_api_check = spark.read.parquet(api_check_path)

display(df_api_check)